In [5]:
import os
import re
import time
import pickle
import platform
import numpy as np
import torch
import torch.nn as nn
from collections import Counter
from torch.utils.data import TensorDataset, DataLoader
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
os.makedirs("saved_models", exist_ok=True)

print("1. Loading CLINC150 dataset...")
dataset = load_dataset("clinc/clinc_oos", "plus")
label_names = dataset["train"].features["intent"].names
oos_label_id = label_names.index("oos")
num_labels = len(label_names)

def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r'\b[a-z]{2,4}-\d{3,4}\b', ' model_code ', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

train_texts = [normalize_text(x["text"]) for x in dataset["train"]]
train_labels = [x["intent"] for x in dataset["train"]]
test_texts = [normalize_text(x["text"]) for x in dataset["test"]]
test_labels = np.array([x["intent"] for x in dataset["test"]])

def evaluate_predictions(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred) * 100
    oos_mask = (y_true == oos_label_id)
    oos_rec = recall_score(oos_mask, (y_pred == oos_label_id)) * 100
    return acc, oos_rec

def get_size_mb(path):
    if os.path.isdir(path):
        total = sum(os.path.getsize(os.path.join(d, f)) for d, _, files in os.walk(path) for f in files)
    else:
        total = os.path.getsize(path)
    return total / (1024 * 1024)

# ==========================================
# MODEL 1: TF-IDF + Logistic Regression [Area B]
# ==========================================
print("2. Training Model 1 (TF-IDF + Logistic Regression)...")
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)
X_train_tfidf = tfidf.fit_transform(train_texts)
X_test_tfidf = tfidf.transform(test_texts)

clf_lr = LogisticRegression(max_iter=1000, C=5.0)
clf_lr.fit(X_train_tfidf, train_labels)

lr_acc, lr_oos = evaluate_predictions(test_labels, clf_lr.predict(X_test_tfidf))
with open("saved_models/model1_tfidf_lr.pkl", "wb") as f:
    pickle.dump({"tfidf": tfidf, "model": clf_lr, "labels": label_names}, f)
lr_size = get_size_mb("saved_models/model1_tfidf_lr.pkl")

sample_inputs = test_texts[:100]
start = time.perf_counter()
for txt in sample_inputs:
    _ = clf_lr.predict(tfidf.transform([txt]))
lr_latency = ((time.perf_counter() - start) / 100) * 1000

# ==========================================
# MODEL 2: Averaged Word Embeddings + FFNN (SGD) [Area A]
# ==========================================
print("3. Training Model 2 (Word Embeddings + Feedforward NN)...")
word_counts = Counter(w for t in train_texts for w in t.split())
vocab = {"<PAD>": 0, "<UNK>": 1}
for word, count in word_counts.items():
    if count >= 2:
        vocab[word] = len(vocab)

MAX_LEN = 32
def encode_words(text):
    ids = [vocab.get(w, 1) for w in text.split()[:MAX_LEN]]
    return ids + [0] * (MAX_LEN - len(ids))

X_train_ids = torch.tensor([encode_words(t) for t in train_texts], dtype=torch.long)
X_test_ids = torch.tensor([encode_words(t) for t in test_texts], dtype=torch.long)
y_train_tensor = torch.tensor(train_labels, dtype=torch.long)

class AveragedEmbeddingFFNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, mode="mean", padding_idx=0)
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, num_classes)
        )
    def forward(self, x):
        return self.fc(self.embedding(x))

nn_model = AveragedEmbeddingFFNN(len(vocab), 128, 256, num_labels)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(nn_model.parameters(), lr=0.05, momentum=0.9)
train_loader = DataLoader(TensorDataset(X_train_ids, y_train_tensor), batch_size=64, shuffle=True)

nn_model.train()
for epoch in range(40):
    for bx, by in train_loader:
        optimizer.zero_grad()
        loss = criterion(nn_model(bx), by)
        loss.backward()
        optimizer.step()

nn_model.eval()
with torch.no_grad():
    nn_preds = torch.argmax(nn_model(X_test_ids), dim=1).numpy()

nn_acc, nn_oos = evaluate_predictions(test_labels, nn_preds)
torch.save({"state_dict": nn_model.state_dict(), "vocab": vocab, "max_len": MAX_LEN}, "saved_models/model2_ffnn.pt")
nn_size = get_size_mb("saved_models/model2_ffnn.pt")

start = time.perf_counter()
with torch.no_grad():
    for txt in sample_inputs:
        _ = nn_model(torch.tensor([encode_words(txt)], dtype=torch.long))
nn_latency = ((time.perf_counter() - start) / 100) * 1000

# ==========================================
# MODEL 3: Fine-tuned Transformer (BERT-Tiny) [Area C]
# ==========================================
print("4. Fine-tuning Model 3 (BERT-Tiny Transformer)...")
MODEL_NAME = "google/bert_uncased_L-2_H-128_A-2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_enc = tokenizer(train_texts, truncation=True, padding=True, max_length=64)

class IntentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
training_args = TrainingArguments(
    output_dir="./bert_checkpoints",
    num_train_epochs=6,
    per_device_train_batch_size=32,
    learning_rate=3e-4,
    weight_decay=0.01,
    logging_steps=500,
    save_strategy="no",
    report_to="none"
)

Trainer(model=bert_model, args=training_args, train_dataset=IntentDataset(train_enc, train_labels)).train()
bert_model.save_pretrained("saved_models/model3_bert")
tokenizer.save_pretrained("saved_models/model3_bert")

bert_model.to("cpu").eval()
bert_preds = []
with torch.no_grad():
    for i in range(0, len(test_texts), 64):
        batch = tokenizer(test_texts[i:i+64], truncation=True, padding=True, max_length=64, return_tensors="pt")
        bert_preds.extend(torch.argmax(bert_model(**batch).logits, dim=1).numpy())

bert_acc, bert_oos = evaluate_predictions(test_labels, np.array(bert_preds))
bert_size = get_size_mb("saved_models/model3_bert")

start = time.perf_counter()
with torch.no_grad():
    for txt in sample_inputs:
        _ = bert_model(**tokenizer(txt, truncation=True, max_length=64, return_tensors="pt"))
bert_latency = ((time.perf_counter() - start) / 100) * 1000

# ==========================================
# MODEL 4: Quantized Transformer (Dynamic INT8) [Area D]
# ==========================================
print("5. Quantizing Model 4 (Dynamic INT8 BERT-Tiny)...")
quantized_bert = torch.ao.quantization.quantize_dynamic(bert_model, {torch.nn.Linear}, dtype=torch.qint8)
torch.save(quantized_bert.state_dict(), "saved_models/model4_quantized_int8.pt")
q_size = get_size_mb("saved_models/model4_quantized_int8.pt")

q_preds = []
with torch.no_grad():
    for i in range(0, len(test_texts), 64):
        batch = tokenizer(test_texts[i:i+64], truncation=True, padding=True, max_length=64, return_tensors="pt")
        q_preds.extend(torch.argmax(quantized_bert(**batch).logits, dim=1).numpy())

q_acc, q_oos = evaluate_predictions(test_labels, np.array(q_preds))

start = time.perf_counter()
with torch.no_grad():
    for txt in sample_inputs:
        _ = quantized_bert(**tokenizer(txt, truncation=True, max_length=64, return_tensors="pt"))
q_latency = ((time.perf_counter() - start) / 100) * 1000

# ==========================================
# RESULTS TABLE (R4)
# ==========================================
print("\n" + "="*75)
print(f"HARDWARE: {platform.processor()} | PyTorch {torch.__version__}")
print("="*75)
print(f"{'Model':<30} | {'Acc (%)':<8} | {'OOS Rec (%)':<11} | {'Size (MB)':<10} | {'Latency (ms)':<12}")
print("-"*75)
print(f"{'1. TF-IDF + Logistic Reg.':<30} | {lr_acc:<8.2f} | {lr_oos:<11.2f} | {lr_size:<10.2f} | {lr_latency:<12.2f}")
print(f"{'2. Word Emb. + FFNN (SGD)':<30} | {nn_acc:<8.2f} | {nn_oos:<11.2f} | {nn_size:<10.2f} | {nn_latency:<12.2f}")
print(f"{'3. BERT-Tiny (FP32)':<30} | {bert_acc:<8.2f} | {bert_oos:<11.2f} | {bert_size:<10.2f} | {bert_latency:<12.2f}")
print(f"{'4. Quantized BERT-Tiny (INT8)':<30} | {q_acc:<8.2f} | {q_oos:<11.2f} | {q_size:<10.2f} | {q_latency:<12.2f}")
print("="*75)

1. Loading CLINC150 dataset...
2. Training Model 1 (TF-IDF + Logistic Regression)...
3. Training Model 2 (Word Embeddings + Feedforward NN)...
4. Fine-tuning Model 3 (BERT-Tiny Transformer)...


Loading weights: 100%|██████████| 39/39 [00:00<00:00, 9008.58it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing fro

Step,Training Loss
500,3.849936
1000,2.001756
1500,1.168513
2000,0.797609
2500,0.618463


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 90.71it/s]


5. Quantizing Model 4 (Dynamic INT8 BERT-Tiny)...


C:\Users\mmoha\AppData\Local\Temp\ipykernel_35276\2301351706.py:189: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_bert = torch.ao.quantization.quantize_dynamic(bert_model, {torch.nn.Linear}, dtype=torch.qint8)



HARDWARE: Intel64 Family 6 Model 140 Stepping 1, GenuineIntel | PyTorch 2.15.0+cpu
Model                          | Acc (%)  | OOS Rec (%) | Size (MB)  | Latency (ms)
---------------------------------------------------------------------------
1. TF-IDF + Logistic Reg.      | 82.55    | 48.10       | 11.90      | 0.41        
2. Word Emb. + FFNN (SGD)      | 80.22    | 57.90       | 1.77       | 0.14        
3. BERT-Tiny (FP32)            | 76.35    | 34.20       | 17.50      | 1.60        
4. Quantized BERT-Tiny (INT8)  | 76.24    | 33.60       | 15.61      | 4.00        
